<a href="https://colab.research.google.com/github/PromyotKatarat/AI_agent_engineering/blob/main/GAIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q U langchain-tavily langgraph langchain-openai langchain-community

In [15]:
# import os
# import re
# import json
# from google.colab import userdata
# from typing import Annotated, TypedDict, List
# from langchain_openai import ChatOpenAI
# from langchain_community.tools.tavily_search import TavilySearchResults
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
# from langchain_core.tools import tool
# from langgraph.graph import StateGraph, START, END
# from langgraph.graph.message import add_messages
# from langgraph.prebuilt import ToolNode

# # =================================================================
# # 1. API Keys & LLM Setup
# # =================================================================
# os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

# MODEL_NAME = "openrouter/free"
# BASE_URL = "https://openrouter.ai/api/v1"

# llm = ChatOpenAI(
#     base_url=BASE_URL,
#     api_key=os.environ["OPENROUTER_API_KEY"],
#     model=MODEL_NAME,
#     temperature=0
# )

# # =================================================================
# # 2. Custom Tools (ระบบดึงข้อมูลและคัดกรองเนื้อหาเว็บ)
# # =================================================================
# @tool
# def fetch_webpage_content(url: str) -> str:
#     """Use this tool to download and read the full text content of a specific webpage URL
#     when standard search snippets are incomplete.
#     """
#     try:
#         loader = WebBaseLoader(url)
#         docs = loader.load()
#         full_text = "\n".join([doc.page_content for doc in docs])
#         return full_text[:15000]
#     except Exception as e:
#         return f"Error fetching webpage: {str(e)}"

# search_tool = TavilySearchResults(max_results=3)
# tools = [search_tool, fetch_webpage_content]
# tool_node = ToolNode(tools)

# llm_with_tools = llm.bind_tools(tools)

# # =================================================================
# # 3. LangGraph Agent State & Architecture
# # =================================================================
# class AgentState(TypedDict):
#     messages: Annotated[list, add_messages]

# def call_model(state: AgentState):
#     messages = state['messages']
#     response = llm_with_tools.invoke(messages)
#     return {"messages": [response]}

# def should_continue(state: AgentState):
#     messages = state['messages']
#     last_message = messages[-1]
#     if last_message.tool_calls:
#         return "tools"
#     return END

# # Build Graph Workflow
# workflow = StateGraph(AgentState)
# workflow.add_node("agent", call_model)
# workflow.add_node("tools", tool_node)

# workflow.add_edge(START, "agent")
# workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
# workflow.add_edge("tools", "agent")

# agent_app = workflow.compile()

# # =================================================================
# # 4. Core Process Function (สวยงาม, Scannable & Debug ง่าย)
# # =================================================================
# def process_task(item: dict) -> int:
#     task_id = item.get('task_id', 'unknown')
#     question = item.get('question', '')

#     print(f"\n[SYSTEM LOG] Model: {MODEL_NAME}")
#     print(f"📝 Processing ID: {task_id[:8]}...")
#     print(f"❓ Question: {question}")

#     print("\n" + "="*60)
#     print(" 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)")
#     print("="*60)

#     # คำสั่งเชิงระบบแบบไร้การฟิกคำตอบล่วงหน้าเพื่อหลีกเลี่ยง Overfitting
#     prompt_instruction = f"""Task: {question}

# System Instructions for Agent:
# 1. Information Retrieval: Search for the most official, comprehensive primary source (such as English Wikipedia) regarding the subject's record or list.
# 2. Section Filtering: Locate the exact section or category that matches the target entity type specified in the question. Do not mix up sub-categories.
# 3. Criteria-Based Evaluation: Strict filtering based on the conditions given in the prompt (e.g., specific year ranges, inclusive boundaries).
# 4. Counting Logic: Count only unique, verified entries that satisfy all conditions.

# Output Format Requirement:
# Analysis: [Provide a brief step-by-step reasoning of how you filtered the entries]
# FINAL_COUNT: [Provide only the final pure integer answer]"""

#     inputs = {"messages": [HumanMessage(content=prompt_instruction)]}
#     final_ans = ""

#     try:
#         # สตรีมลูปและจัดระเบียบการ Print หน้าจอแยกสัดส่วนให้ Debug ง่าย
#         for chunk in agent_app.stream(inputs, config={"recursion_limit": 25}):
#             for node_name, node_data in chunk.items():
#                 if "messages" in node_data and node_data["messages"]:
#                     last_msg = node_data["messages"][-1]

#                     # 4.1 ตกแต่งการแสดงผลของฝั่ง TOOLS ให้กระชับ เห็นลิงก์อ้างอิงชัดเจน
#                     if node_name == "tools":
#                         print(f"\n⚙️  [Node: {node_name.upper()}] -> Executing Information Retrieval...")
#                         print("-" * 60)

#                         # แยกตรวจสอบโครงสร้างผลการเสิร์ช
#                         if isinstance(last_msg.content, str) and last_msg.content.startswith("[{"):
#                             try:
#                                 search_results = json.loads(last_msg.content)
#                                 print("🌐 Active Search References Extracted:")
#                                 for res in search_results:
#                                     title = res.get('title', 'Unknown Source')
#                                     url = res.get('url', '#')
#                                     print(f" 📄 Source Verified: {title} ({url})")
#                             except:
#                                 print(f"📄 Data Fetched (Raw Snippet View):\n{last_msg.content[:300]}...")
#                         else:
#                             print(f"📥 Content Successfully Ingested from Target URL")
#                             print(f"   [Data Length: {len(last_msg.content)} characters parsed]")
#                         print("-" * 60)

#                     # 4.2 ตกแต่งการแสดงผลฝั่ง AGENT เพื่อแกะตรรกะการคิดเลข
#                     elif node_name == "agent" and last_msg.content:
#                         print(f"\n🤖 [Node: {node_name.upper()}] -> Evaluating Evidence")
#                         print("-" * 60)
#                         print(last_msg.content.strip())
#                         print("-" * 60)

#                         final_ans = last_msg.content

#         # =================================================================
#         # 5. Score Extraction & Fallback Engine
#         # =================================================================
#         print("\n" + "="*60)
#         print(" 🏁 SCORE EXTRACTION")
#         print("="*60)

#         match = re.search(r"FINAL_COUNT:\s*(\d+)", final_ans)
#         if match:
#             extracted_number = int(match.group(1))
#             print(f"🎯 Successfully Extracted Score: {extracted_number}")
#             return extracted_number
#         else:
#             print("⚠️ Warning: Could not find 'FINAL_COUNT: [number]' pattern.")
#             last_line = final_ans.strip().split('\n')[-1]
#             digits = re.findall(r"\d+", last_line)
#             if digits:
#                 fallback_number = int(digits[-1])
#                 print(f"🔄 Fallback Extracted Score from last line: {fallback_number}")
#                 return fallback_number

#             print("❌ Failure: No numeric answer could be extracted.")
#             return 0

#     except Exception as e:
#         print(f"❌ Error occurred during ReAct Execution: {str(e)}")
#         return 0

# # =================================================================
# # 6. Test Driver (ทดสอบรันผลจริง)
# # =================================================================
# if __name__ == "__main__":
#     # กล่องจำลองสำหรับป้อนโจทย์ทดสอบความถูกต้อง
#     sample_item = {
#         'task_id': '8e867cd7-demo',
#         'question': 'How many studio albums were published by Mercedes Sosa between 1993 and 1999 (included)? You can use the latest 2022 version of english wikipedia.'
#     }

#     # รันการทำงาน
#     output_score = process_task(sample_item)

#     print("\n" + "#"*50)
#     print(f"🚀 FINAL RE-RUN EVALUATION RESULT: {output_score}")
#     print("#"*50)


[SYSTEM LOG] Model: openrouter/free
📝 Processing ID: 8e867cd7...
❓ Question: How many studio albums were published by Mercedes Sosa between 1993 and 1999 (included)? You can use the latest 2022 version of english wikipedia.

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)
❌ Error occurred during ReAct Execution: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1784160000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free 

In [17]:
!pip install -q langchain-google-genai langgraph langchain-community tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.6 MB/s eta 0:00:00


In [29]:
# import os
# import re
# import json
# import time
# from google.colab import userdata
# from typing import Annotated, TypedDict, List
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_community.tools.tavily_search import TavilySearchResults
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
# from langchain_core.tools import tool
# from langgraph.graph import StateGraph, START, END
# from langgraph.graph.message import add_messages
# from langgraph.prebuilt import ToolNode

# # =================================================================
# # 1. API Keys & LLM Setup (คอนฟิกใช้ Gemini 3.1 Flash Lite)
# # =================================================================
# os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# MODEL_NAME = "gemini-3.1-flash-lite"

# llm = ChatGoogleGenerativeAI(
#     model=MODEL_NAME,
#     temperature=0
# )

# # =================================================================
# # 2. Custom Tools
# # =================================================================
# @tool
# def fetch_webpage_content(url: str) -> str:
#     """Use this tool to download and read the full text content of a specific webpage URL
#     when standard search snippets are incomplete.
#     """
#     try:
#         loader = WebBaseLoader(url)
#         docs = loader.load()
#         full_text = "\n".join([doc.page_content for doc in docs])
#         return full_text[:15000]
#     except Exception as e:
#         return f"Error fetching webpage: {str(e)}"

# search_tool = TavilySearchResults(max_results=3)
# tools = [search_tool, fetch_webpage_content]
# tool_node = ToolNode(tools)

# llm_with_tools = llm.bind_tools(tools)

# # =================================================================
# # 3. LangGraph Agent State & Architecture + Rate Limiter Configuration
# # =================================================================
# class AgentState(TypedDict):
#     messages: Annotated[list, add_messages]

# def call_model(state: AgentState):
#     print("\n⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...")
#     time.sleep(4)

#     messages = state['messages']
#     response = llm_with_tools.invoke(messages)
#     return {"messages": [response]}

# def should_continue(state: AgentState):
#     messages = state['messages']
#     last_message = messages[-1]
#     if last_message.tool_calls:
#         return "tools"
#     return END

# # Build Graph Workflow
# workflow = StateGraph(AgentState)
# workflow.add_node("agent", call_model)
# workflow.add_node("tools", tool_node)

# workflow.add_edge(START, "agent")
# workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
# workflow.add_edge("tools", "agent")

# agent_app = workflow.compile()

# # =================================================================
# # 4. Core Process Function (แก้ไขบั๊กแปลงข้อความให้รองรับ List)
# # =================================================================
# def process_task(item: dict) -> int:
#     task_id = item.get('task_id', 'unknown')
#     question = item.get('question', '')

#     print(f"\n[SYSTEM LOG] Model: {MODEL_NAME}")
#     print(f"📝 Processing ID: {task_id[:8]}...")
#     print(f"❓ Question: {question}")

#     print("\n" + "="*60)
#     print(" 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)")
#     print("="*60)

#     prompt_instruction = f"""Task: {question}

# System Instructions for Agent:
# 1. Information Retrieval: Search for the most official, comprehensive primary source (such as English Wikipedia) regarding the subject's record or list.
# 2. Section Filtering: Locate the exact section or category that matches the target entity type specified in the question. Do not mix up sub-categories.
# 3. Criteria-Based Evaluation: Strict filtering based on the conditions given in the prompt (e.g., specific year ranges, inclusive boundaries).
# 4. Counting Logic: Count only unique, verified entries that satisfy all conditions.

# Output Format Requirement:
# Analysis: [Provide a brief step-by-step reasoning of how you filtered the entries]
# FINAL_COUNT: [Provide only the final pure integer answer]"""

#     inputs = {"messages": [HumanMessage(content=prompt_instruction)]}
#     final_ans = ""

#     try:
#         for chunk in agent_app.stream(inputs, config={"recursion_limit": 20}):
#             for node_name, node_data in chunk.items():
#                 if "messages" in node_data and node_data["messages"]:
#                     last_msg = node_data["messages"][-1]

#                     # ป้องกันกรณีเนื้อหาถูกส่งออกมาเป็นรูปแบบอื่นที่ไม่ใช่ String
#                     msg_content = ""
#                     if isinstance(last_msg.content, str):
#                         msg_content = last_msg.content
#                     elif isinstance(last_msg.content, list):
#                         msg_content = "\n".join([str(x) for x in last_msg.content])

#                     if node_name == "tools":
#                         print(f"\n⚙️  [Node: {node_name.upper()}] -> Executing Information Retrieval...")
#                         print("-" * 60)

#                         if msg_content.startswith("[{"):
#                             try:
#                                 search_results = json.loads(msg_content)
#                                 print("🌐 Active Search References Extracted:")
#                                 for res in search_results:
#                                     title = res.get('title', 'Unknown Source')
#                                     url = res.get('url', '#')
#                                     print(f" 📄 Source Verified: {title} ({url})")
#                             except:
#                                 print(f"📄 Data Fetched (Raw Snippet View):\n{msg_content[:300]}...")
#                         else:
#                             print(f"📥 Content Successfully Ingested from Target URL")
#                             print(f"   [Data Length: {len(msg_content)} characters parsed]")
#                         print("-" * 60)

#                     elif node_name == "agent" and msg_content:
#                         print(f"\n🤖 [Node: {node_name.upper()}] -> Evaluating Evidence")
#                         print("-" * 60)

#                         # แก้ปัญหาการแสดงผลหลุดรูปแบบ: แปลงโค้ด \n ให้กลายเป็นการขึ้นบรรทัดใหม่จริง ๆ
#                         clean_content = msg_content.replace('\\n', '\n').strip()

#                         # ตกแต่งหัวข้อให้ Scannable สวยงามขึ้นอัตโนมัติ
#                         clean_content = clean_content.replace('Analysis:', '📋 **Analysis Summary**')
#                         clean_content = clean_content.replace('FINAL_COUNT:', '🎯 **FINAL_COUNT**:')

#                         print(clean_content)
#                         print("-" * 60)

#                         final_ans = msg_content

#         # =================================================================
#         # 5. Score Extraction & Fallback Engine
#         # =================================================================
#         print("\n" + "="*60)
#         print(" 🏁 SCORE EXTRACTION")
#         print("="*60)

#         match = re.search(r"FINAL_COUNT:\s*(\d+)", final_ans)
#         if match:
#             extracted_number = int(match.group(1))
#             print(f"🎯 Successfully Extracted Score: {extracted_number}")
#             return extracted_number
#         else:
#             print("⚠️ Warning: Could not find 'FINAL_COUNT: [number]' pattern.")
#             last_line = final_ans.strip().split('\n')[-1]
#             digits = re.findall(r"\d+", last_line)
#             if digits:
#                 fallback_number = int(digits[-1])
#                 print(f"🔄 Fallback Extracted Score from last line: {fallback_number}")
#                 return fallback_number

#             print("❌ Failure: No numeric answer could be extracted.")
#             return 0

#     except Exception as e:
#         print(f"❌ Error occurred during ReAct Execution: {str(e)}")
#         return 0

# # =================================================================
# # 6. Test Driver
# # =================================================================
# if __name__ == "__main__":
#     sample_item = {
#         'task_id': 'gemini3.1-lite-test',
#         'question': 'How many studio albums were published by Mercedes Sosa between 1993 and 1999 (included)? You can use the latest 2022 version of english wikipedia.'
#     }

#     output_score = process_task(sample_item)

#     print("\n" + "#"*50)
#     print(f"🚀 FINAL RE-RUN EVALUATION RESULT: {output_score}")
#     print("#"*50)


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: gemini3....
❓ Question: How many studio albums were published by Mercedes Sosa between 1993 and 1999 (included)? You can use the latest 2022 version of english wikipedia.

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
🌐 Active Search References Extracted:
 📄 Source Verified: Cantora, un Viaje Íntimo (https://en.wikipedia.org/wiki/Cantora,_un_Viaje_%C3%8Dntimo)
 📄 Source Verified: Mercedes Sosa (https://en.wikipedia.org/wiki/Mercedes_Sosa)
 📄 Source Verified: Mercedes Sosa - Wikipedia, la enciclopedia libre (https://es.wikipedia.org/wiki/Mercedes_Sosa)
------------------------------------------------------------

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
--------------

In [26]:
# import os
# import re
# import json
# import time
# import urllib.request
# from google.colab import userdata
# from typing import Annotated, TypedDict, List
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_community.tools.tavily_search import TavilySearchResults
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
# from langchain_core.tools import tool
# from langgraph.graph import StateGraph, START, END
# from langgraph.graph.message import add_messages
# from langgraph.prebuilt import ToolNode

# # =================================================================
# # 1. API Keys & LLM Setup (คอนฟิก Gemini 3.1 Flash Lite)
# # =================================================================
# os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# MODEL_NAME = "gemini-3.1-flash-lite"

# llm = ChatGoogleGenerativeAI(
#     model=MODEL_NAME,
#     temperature=0
# )

# # =================================================================
# # 2. Custom Tools (เพิ่มระบบดึง Meta ข้อมูลจาก YouTube)
# # =================================================================
# @tool
# def fetch_youtube_data(url: str) -> str:
#     """Use this tool when the question asks about a specific YouTube video URL.
#     It extracts metadata to help identify the content of the video.
#     """
#     try:
#         # ดึง oEmbed Data พื้นฐานเพื่อดู Title และข้อมูลเบื้องต้นของคลิป
#         oembed_url = f"https://www.youtube.com/oembed?url={url}&format=json"
#         with urllib.request.urlopen(oembed_url, timeout=5) as response:
#             data = json.loads(response.read().decode())
#             return f"Video Title: {data.get('title')}\nAuthor: {data.get('author_name')}\nDescription context extracted."
#     except Exception as e:
#         return f"Could not fetch YouTube metadata directly: {str(e)}. Please fallback to standard search or reasoning."

# @tool
# def fetch_webpage_content(url: str) -> str:
#     """Use this tool to download and read the full text content of a specific webpage URL
#     when standard search snippets are incomplete.
#     """
#     try:
#         loader = WebBaseLoader(url)
#         docs = loader.load()
#         full_text = "\n".join([doc.page_content for doc in docs])
#         return full_text[:15000]
#     except Exception as e:
#         return f"Error fetching webpage: {str(e)}"

# search_tool = TavilySearchResults(max_results=3)

# # รวมเครื่องมือทั้งหมดส่งให้ Agent
# tools = [search_tool, fetch_webpage_content, fetch_youtube_data]
# tool_node = ToolNode(tools)

# llm_with_tools = llm.bind_tools(tools)

# # =================================================================
# # 3. LangGraph Agent State & Architecture + Rate Limiter Configuration
# # =================================================================
# class AgentState(TypedDict):
#     messages: Annotated[list, add_messages]

# def call_model(state: AgentState):
#     # ป้องกัน Rate limit 15 RPM ของโมเดล Lite
#     print("\n⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...")
#     time.sleep(4)

#     messages = state['messages']
#     response = llm_with_tools.invoke(messages)
#     return {"messages": [response]}

# def should_continue(state: AgentState):
#     messages = state['messages']
#     last_message = messages[-1]
#     if last_message.tool_calls:
#         return "tools"
#     return END

# # Build Graph Workflow
# workflow = StateGraph(AgentState)
# workflow.add_node("agent", call_model)
# workflow.add_node("tools", tool_node)

# workflow.add_edge(START, "agent")
# workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
# workflow.add_edge("tools", "agent")

# agent_app = workflow.compile()

# # =================================================================
# # 4. Core Process Function (จัดหน้าจอสวยงาม + แก้อาการอ่านข้อความแบบ List)
# # =================================================================
# def process_task(item: dict) -> int:
#     task_id = item.get('task_id', 'unknown')
#     question = item.get('question', '')

#     print(f"\n[SYSTEM LOG] Model: {MODEL_NAME}")
#     print(f"📝 Processing ID: {task_id[:8]}...")
#     print(f"❓ Question: {question}")

#     print("\n" + "="*60)
#     print(" 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)")
#     print("="*60)

#     prompt_instruction = f"""Task: {question}

# System Instructions for Agent:
# 1. Information Retrieval: If a video URL is provided, try extracting its details or searching for analyses, transcripts, or breakdowns of that specific video scene online.
# 2. Content Filtering: Pinpoint the exact visual details requested (e.g., number of bird species on camera simultaneously).
# 3. Counting Logic: Identify each unique entity type present at the peak moment and count them accurately.

# Output Format Requirement:
# Analysis: [Provide a brief step-by-step reasoning of how you analyzed the scene or data]
# FINAL_COUNT: [Provide only the final pure integer answer]"""

#     inputs = {"messages": [HumanMessage(content=prompt_instruction)]}
#     final_ans = ""

#     try:
#         for chunk in agent_app.stream(inputs, config={"recursion_limit": 20}):
#             for node_name, node_data in chunk.items():
#                 if "messages" in node_data and node_data["messages"]:
#                     last_msg = node_data["messages"][-1]

#                     msg_content = ""
#                     if isinstance(last_msg.content, str):
#                         msg_content = last_msg.content
#                     elif isinstance(last_msg.content, list):
#                         msg_content = "\n".join([str(x) for x in last_msg.content])

#                     if node_name == "tools":
#                         print(f"\n⚙️  [Node: {node_name.upper()}] -> Executing Information Retrieval...")
#                         print("-" * 60)

#                         if msg_content.startswith("[{"):
#                             try:
#                                 search_results = json.loads(msg_content)
#                                 print("🌐 Active Search References Extracted:")
#                                 for res in search_results:
#                                     title = res.get('title', 'Unknown Source')
#                                     url = res.get('url', '#')
#                                     print(f" 📄 Source Verified: {title} ({url})")
#                             except:
#                                 print(f"📄 Data Fetched (Raw Snippet View):\n{msg_content[:300]}...")
#                         else:
#                             print(f"📥 Content Ingested Metadata / Web Content Output:")
#                             print(f"   {msg_content.strip()}")
#                         print("-" * 60)

#                     elif node_name == "agent" and msg_content:
#                         print(f"\n🤖 [Node: {node_name.upper()}] -> Evaluating Evidence")
#                         print("-" * 60)

#                         # แยกบรรทัดและแต่งหัวข้อให้อ่านง่าย Scannable
#                         clean_content = msg_content.replace('\\n', '\n').strip()
#                         clean_content = clean_content.replace('Analysis:', '📋 **Analysis Summary**')
#                         clean_content = clean_content.replace('FINAL_COUNT:', '🎯 **FINAL_COUNT**:')

#                         print(clean_content)
#                         print("-" * 60)

#                         final_ans = msg_content

#         # =================================================================
#         # 5. Score Extraction & Fallback Engine
#         # =================================================================
#         print("\n" + "="*60)
#         print(" 🏁 SCORE EXTRACTION")
#         print("="*60)

#         match = re.search(r"FINAL_COUNT:\s*(\d+)", final_ans)
#         if match:
#             extracted_number = int(match.group(1))
#             print(f"🎯 Successfully Extracted Score: {extracted_number}")
#             return extracted_number
#         else:
#             print("⚠️ Warning: Could not find 'FINAL_COUNT: [number]' pattern.")
#             last_line = final_ans.strip().split('\n')[-1]
#             digits = re.findall(r"\d+", last_line)
#             if digits:
#                 fallback_number = int(digits[-1])
#                 print(f"🔄 Fallback Extracted Score from last line: {fallback_number}")
#                 return fallback_number

#             print("❌ Failure: No numeric answer could be extracted.")
#             return 0

#     except Exception as e:
#         print(f"❌ Error occurred during ReAct Execution: {str(e)}")
#         return 0

# # =================================================================
# # 6. Test Driver (รันโจทย์ข้อที่ 2 ของคุณทันที)
# # =================================================================
# if __name__ == "__main__":
#     sample_item = {
#         'task_id': 'youtube-birds-task',
#         'question': 'In the video https://www.youtube.com/watch?v=L1vXCYZAYYM, what is the highest number of bird species to be on camera simultaneously?'
#     }

#     output_score = process_task(sample_item)

#     print("\n" + "#"*50)
#     print(f"🚀 FINAL RE-RUN EVALUATION RESULT: {output_score}")
#     print("#"*50)


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: youtube-...
❓ Question: In the video https://www.youtube.com/watch?v=L1vXCYZAYYM, what is the highest number of bird species to be on camera simultaneously?

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
📥 Content Ingested Metadata / Web Content Output:
   Video Title: Penguin Chicks Stand Up To Giant Petrel...With The Help of a Friend!
Author: John Downer Productions
Description context extracted.
------------------------------------------------------------

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
🌐 Active Search References Extracted:
 📄 Source Verified: Will Robotic Spy Chick Become The Giant Petrel's Ne

In [55]:
import os
import re
import json
import time
import urllib.request
from google.colab import userdata
from typing import Annotated, TypedDict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# =================================================================
# 1. API Keys & LLM Setup (Gemini 3.1 Flash Lite)
# =================================================================
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

MODEL_NAME = "gemini-3.1-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0
)

# =================================================================
# 2. Custom Multi-Tools Suite
# =================================================================
@tool
def fetch_youtube_data(url: str) -> str:
    """Use this tool when the question asks about a specific YouTube video URL.
    It extracts metadata to help identify the content of the video.
    """
    try:
        oembed_url = f"https://www.youtube.com/oembed?url={url}&format=json"
        with urllib.request.urlopen(oembed_url, timeout=5) as response:
            data = json.loads(response.read().decode())
            return f"Video Title: {data.get('title')}\nAuthor: {data.get('author_name')}\nMetadata captured successfully."
    except Exception as e:
        return f"Could not fetch YouTube metadata directly: {str(e)}. Fallback to search."

@tool
def fetch_webpage_content(url: str) -> str:
    """Use this tool to download and read the full text content of a specific webpage URL
    when standard search snippets are incomplete.
    """
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()
        full_text = "\n".join([doc.page_content for doc in docs])
        return full_text[:15000]
    except Exception as e:
        return f"Error fetching webpage: {str(e)}"

# 1. แทรกฟังก์ชันนี้ต่อท้ายลงไปในเซลล์ 29
@tool
def solve_commutative_counterexamples(table_str: str) -> str:
    """Useful for finding subsets of elements that violate the commutative property (x * y != y * x)
    given a markdown table representation of an algebraic binary operation.
    """
    try:
        lines = [line.strip() for line in table_str.strip().split('\n') if line.strip()]
        table_lines = [l for l in lines if '|' in l and '---' not in l]
        if not table_lines:
            return "Error: Invalid table format."

        headers = [x.strip() for x in table_lines[0].split('|')[2:-1]]
        n = len(headers)

        matrix = {}
        for line in table_lines[1:]:
            parts = [x.strip() for x in line.split('|')[1:-1]]
            row_header = parts[0]
            row_values = parts[1:]
            matrix[row_header] = {headers[i]: row_values[i] for i in range(min(len(headers), len(row_values)))}

        counter_examples_set = set()
        for i in range(n):
            for j in range(i + 1, n):
                element_i = headers[i]
                element_j = headers[j]
                val_ij = matrix.get(element_i, {}).get(element_j, "")
                val_ji = matrix.get(element_j, {}).get(element_i, "")
                if val_ij != val_ji:
                    counter_examples_set.add(element_i)
                    counter_examples_set.add(element_j)

        result_list = sorted(list(counter_examples_set))
        final_answer = ", ".join(result_list)
        return f"Analysis: Calculated via programmatic matrix scan.\nFINAL_COUNT: 0\nResult Set: {final_answer}"
    except Exception as e:
        return f"Error scanning matrix: {str(e)}"

# 2. แก้ไขบรรทัดนี้เพิ่มชื่อ Tool ใหม่เข้าไป
search_tool = TavilySearchResults(max_results=3)
tools = [search_tool, fetch_webpage_content, solve_commutative_counterexamples] # <-- เพิ่มตรงนี้
tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

# =================================================================
# 3. LangGraph Agent Architecture & Rate Limiter (15 RPM)
# =================================================================
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state: AgentState):
    print("\n⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...")
    time.sleep(4)

    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: AgentState):
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")
agent_app = workflow.compile()

# =================================================================
# 4. Core Process Engine (ปรับปรุงการแสดงผล Tools Log ไม่ให้ข้อความยาวล้นหน้าจอ)
# =================================================================
def process_task(item: dict) -> int:
    task_id = item.get('task_id', 'unknown')
    question = item.get('question', '')

    print(f"\n[SYSTEM LOG] Model: {MODEL_NAME}")
    print(f"📝 Processing ID: {task_id}")
    print(f"❓ Question: {question}")

    print("\n" + "="*60)
    print(" 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)")
    print("="*60)

    prompt_instruction = f"""Task: {question}

System Instructions for Agent:
1. Information Retrieval: Dynamically leverage available tools (Search, Web fetcher, or YouTube fetcher) to collect verified ground truth.
2. Strict Type Filtering: For discography tasks, look up the exact "Studio albums" list. DO NOT count compilation albums, live albums, or collaborations unless explicitly stated as a core studio album. Cross-check years carefully.
3. Counting Logic: Identify each unique item matching the criteria and provide a meticulous breakdown.

Output Format Requirement:
Analysis: [Provide a brief step-by-step reasoning or summary table]
FINAL_COUNT: [Provide only the final pure integer answer]"""

    inputs = {"messages": [HumanMessage(content=prompt_instruction)]}
    final_ans = ""

    try:
        for chunk in agent_app.stream(inputs, config={"recursion_limit": 20}):
            for node_name, node_data in chunk.items():
                if "messages" in node_data and node_data["messages"]:
                    last_msg = node_data["messages"][-1]

                    # --- แก้บั๊กแปลงชนิดข้อมูล Object ให้เป็น String ---
                    msg_content = ""
                    if isinstance(last_msg.content, str):
                        msg_content = last_msg.content
                    elif isinstance(last_msg.content, list):
                        if len(last_msg.content) > 0 and isinstance(last_msg.content[0], dict):
                            msg_content = last_msg.content[0].get('text', str(last_msg.content))
                        else:
                            msg_content = "\n".join([str(x) for x in last_msg.content])
                    elif isinstance(last_msg.content, dict):
                        msg_content = last_msg.content.get('text', str(last_msg.content))

                    # --- [แก้ไขจุดนี้] แสดงผล Log แบบจำกัดความยาวเพื่อความสวยงาม ---
                    if node_name == "tools":
                        print(f"\n⚙️  [Node: {node_name.upper()}] -> Executing Information Retrieval...")
                        print("-" * 60)

                        if msg_content.startswith("[{"):
                            try:
                                search_results = json.loads(msg_content)
                                print("🌐 Active Search References Extracted:")
                                for res in search_results:
                                    print(f" 📄 Source Verified: {res.get('title')} ({res.get('url')})")
                            except:
                                print(f"📄 Data Fetched (Snippet View):\n{msg_content[:300]}...")
                        else:
                            # ครอบคลุมเคสโหลดหน้าเว็บยาวๆ (เช่น Wikipedia) ให้แสดงตัวอย่างสั้นๆ แทนพ่นออกทั้งหมด
                            print(f"📥 Content Ingested Successfully! [Total length: {len(msg_content)} characters]")
                            print(f"📄 Preview text:\n{msg_content.strip()[:300]}...")
                        print("-" * 60)

                    elif node_name == "agent" and msg_content:
                        print(f"\n🤖 [Node: {node_name.upper()}] -> Evaluating Evidence")
                        print("-" * 60)

                        # จัดระเบียบข้อความ เปลี่ยนแท็ก \n ให้ขึ้นบรรทัดใหม่ และเน้นหัวข้อหลัก
                        clean_content = msg_content.replace('\\n', '\n').strip()
                        clean_content = clean_content.replace('Analysis:', '📋 **Analysis Summary**')
                        clean_content = clean_content.replace('FINAL_COUNT:', '🎯 **FINAL_COUNT**:')

                        print(clean_content)
                        print("-" * 60)

                        final_ans = msg_content

       # =================================================================
        # 5. Score Extraction & Fallback Engine
        # =================================================================
        print("\n" + "="*60)
        print(" 🏁 SCORE EXTRACTION")
        print("="*60)

        match = re.search(r"FINAL_COUNT:\s*(.+)", final_ans)
        if match:
            extracted_val = match.group(1).strip()
            print(f"🎯 Successfully Extracted Score: {extracted_val}")
            return extracted_val
        else:
            print("⚠️ Warning: Could not find 'FINAL_COUNT' pattern. Engaging Fallback Engine...")
            lines = final_ans.strip().split('\n')
            for line in reversed(lines):
                if ":" in line:
                    fallback_val = line.split(":")[-1].strip()
                    if fallback_val:
                        print(f"🔄 Fallback Extracted Score: {fallback_val}")
                        return fallback_val
            return final_ans.strip()

    except Exception as e:
        print(f"❌ Error occurred during ReAct Execution: {str(e)}")
        return 0

# =================================================================
# 6. Test Driver (ปิดตัวรันเก่า ปล่อยผ่านปลอดภัย)
# =================================================================
if __name__ == "__main__":
    pass


In [38]:
import os
import re
import json
import time
import urllib.request
import requests
from PIL import Image
from io import BytesIO
from google.colab import userdata
from typing import Annotated, TypedDict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# =================================================================
# 1. API Keys & LLM Setup (Gemini 3.1 Flash Lite)
# =================================================================
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

MODEL_NAME = "gemini-3.1-flash-lite"

# กำหนดให้ใช้ Gemini ตัวเดิมที่มีความสามารถในการเข้าใจรูปภาพได้ในตัว
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0
)

# =================================================================
# 2. Custom Multi-Tools Suite (เพิ่ม Vision Tool สำหรับข้อ 4)
# =================================================================
@tool
def analyze_chess_image_url(image_url: str) -> str:
    """Use this tool when you encounter a chess position image URL.
    It downloads the image and leverages Gemini's vision capability to analyze
    the board setup and determine the best winning move in algebraic notation.
    """
    try:
        # 1. ดาวน์โหลดรูปภาพจาก Server ของโจทย์
        response = requests.get(image_url, timeout=10)
        img = Image.open(BytesIO(response.content))

        # 2. เรียกใช้งาน Vision API ของ Gemini โดยตรงผ่านโครงสร้าง Message เพื่อวิเคราะห์กระดาน
        vision_llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0)
        instruction = (
            "You are a chess grandmaster. Review the chess position provided in the image. "
            "It is black's turn. Provide the correct next move for black which guarantees a win. "
            "Please analyze carefully and provide your final answer in standard algebraic notation (e.g., Rd5, Nf3)."
        )

        message = HumanMessage(
            content=[
                {"type": "text", "text": instruction},
                {"type": "image_url", "image_url": image_url}
            ]
        )

        res = vision_llm.invoke([message])
        return f"Chess Vision Analysis Result:\n{res.content}"
    except Exception as e:
        return f"Error analyzing chess image: {str(e)}. Please check if the URL is accessible."

@tool
def fetch_youtube_data(url: str) -> str:
    """Use this tool when the question asks about a specific YouTube video URL."""
    try:
        oembed_url = f"https://www.youtube.com/oembed?url={url}&format=json"
        with urllib.request.urlopen(oembed_url, timeout=5) as response:
            data = json.loads(response.read().decode())
            return f"Video Title: {data.get('title')}\nAuthor: {data.get('author_name')}\nMetadata captured successfully."
    except Exception as e:
        return f"Could not fetch YouTube metadata: {str(e)}."

@tool
def fetch_webpage_content(url: str) -> str:
    """Use this tool to download and read the full text content of a specific webpage URL."""
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()
        full_text = "\n".join([doc.page_content for doc in docs])
        return full_text[:15000]
    except Exception as e:
        return f"Error fetching webpage: {str(e)}"

search_tool = TavilySearchResults(max_results=3)

# ผูกเครื่องมือทั้ง 4 ชิ้นเข้าด้วยกัน
tools = [search_tool, fetch_webpage_content, fetch_youtube_data, analyze_chess_image_url]
tool_node = ToolNode(tools)

llm_with_tools = llm.bind_tools(tools)

# =================================================================
# 3. LangGraph Agent Architecture & Rate Limiter
# =================================================================
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state: AgentState):
    print("\n⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...")
    time.sleep(4)

    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: AgentState):
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")
agent_app = workflow.compile()

# =================================================================
# 4. Core Process Engine (จำกัดความยาว Output ไม่ให้รกหน้าจอ)
# =================================================================
def process_task(item: dict) -> str:
    task_id = item.get('task_id', 'unknown')
    question = item.get('question', '')

    print(f"\n[SYSTEM LOG] Model: {MODEL_NAME}")
    print(f"📝 Processing ID: {task_id}")
    print(f"❓ Question: {question}")

    print("\n" + "="*60)
    print(" 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)")
    print("="*60)

    prompt_instruction = f"""Task: {question}

System Instructions for Agent:
1. Information Retrieval: Dynamically leverage available tools. If the prompt specifies a chess image URL, call 'analyze_chess_image_url' immediately.
2. Logic & Analysis: Process the facts meticulously based on the tool outputs.
3. Output Format: Try to output the format as:
Analysis: [Brief breakdown]
FINAL_COUNT: [The chess move string or number]"""

    inputs = {"messages": [HumanMessage(content=prompt_instruction)]}
    final_ans = ""

    try:
        for chunk in agent_app.stream(inputs, config={"recursion_limit": 20}):
            for node_name, node_data in chunk.items():
                if "messages" in node_data and node_data["messages"]:
                    last_msg = node_data["messages"][-1]

                    msg_content = ""
                    if isinstance(last_msg.content, str):
                        msg_content = last_msg.content
                    elif isinstance(last_msg.content, list):
                        if len(last_msg.content) > 0 and isinstance(last_msg.content[0], dict):
                            msg_content = last_msg.content[0].get('text', str(last_msg.content))
                        else:
                            msg_content = "\n".join([str(x) for x in last_msg.content])
                    elif isinstance(last_msg.content, dict):
                        msg_content = last_msg.content.get('text', str(last_msg.content))

                    if node_name == "tools":
                        print(f"\n⚙️  [Node: {node_name.upper()}] -> Executing Information Retrieval...")
                        print("-" * 60)
                        if msg_content.startswith("[{"):
                            try:
                                search_results = json.loads(msg_content)
                                print("🌐 Active Search References Extracted:")
                                for res in search_results:
                                    print(f" 📄 Source Verified: {res.get('title')} ({res.get('url')})")
                            except:
                                print(f"📄 Data Fetched (Snippet View):\n{msg_content[:300]}...")
                        else:
                            print(f"📥 Content Ingested Successfully! [Total length: {len(msg_content)} characters]")
                            print(f"📄 Preview text:\n{msg_content.strip()[:300]}...")
                        print("-" * 60)

                    elif node_name == "agent" and msg_content:
                        print(f"\n🤖 [Node: {node_name.upper()}] -> Evaluating Evidence")
                        print("-" * 60)
                        clean_content = msg_content.replace('\\n', '\n').strip()
                        clean_content = clean_content.replace('Analysis:', '📋 **Analysis Summary**')
                        clean_content = clean_content.replace('FINAL_COUNT:', '🎯 **FINAL_COUNT**:')
                        print(clean_content)
                        print("-" * 60)
                        final_ans = msg_content

        print("\n" + "="*60)
        print(" 🏁 SCORE EXTRACTION")
        print("="*60)

        # ปรับการดึงข้อมูลตอนท้ายให้ยืดหยุ่นรองรับตัวอักษรตาเดินหมากรุก (เช่น Rd5)
        match = re.search(r"FINAL_COUNT:\s*(\S+)", final_ans)
        if match:
            extracted_ans = match.group(1).strip()
            print(f"🎯 Successfully Extracted Answer: {extracted_ans}")
            return extracted_ans
        else:
            lines = final_ans.strip().split('\n')
            last_line = lines[-1] if lines else ""
            print(f"🔄 Fallback Extracted Answer from last line: {last_line}")
            return last_line

    except Exception as e:
        print(f"❌ Error occurred: {str(e)}")
        return "Error"

# =================================================================
# 5. Test Driver (ยิงเข้าหาไฟล์ภาพของโจทย์ข้อที่ 4 โดยตรง)
# =================================================================
if __name__ == "__main__":
    task_4 = {
        'task_id': 'cca530fc-4052-43b2-b130-b30968d8aa44',
        'question': 'Review the chess position provided in the image https://agents-course-unit4-scoring.hf.space/files/cca530fc-4052-43b2-b130-b30968d8aa44.png. It is black\'s turn. Provide the correct next move for black which guarantees a win. Please provide your response in algebraic notation.'
    }

    final_move = process_task(task_4)
    print(f"\n🚀 FINAL MOVE FOR BLACK: {final_move}")


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: cca530fc-4052-43b2-b130-b30968d8aa44
❓ Question: Review the chess position provided in the image https://agents-course-unit4-scoring.hf.space/files/cca530fc-4052-43b2-b130-b30968d8aa44.png. It is black's turn. Provide the correct next move for black which guarantees a win. Please provide your response in algebraic notation.

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
📥 Content Ingested Successfully! [Total length: 134 characters]
📄 Preview text:
Error analyzing chess image: cannot identify image file <_io.BytesIO object at 0x7e13d26dad90>. Please check if the URL is accessible....
------------------------------------------------------------

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Inform

In [39]:
if __name__ == "__main__":
    task_11 = {
        'task_id': 'polish-raymond-magda-m',
        'question': 'Who did the actor who played Ray in the Polish-language version of Everybody Loves Raymond play in Magda M.? Give only the first name.'
    }

    # รันกระบวนการสืบค้นผ่าน ReAct Engine ตัวเดิม
    final_answer = process_task(task_11)
    print(f"\n🚀 FINAL CHARACTER FIRST NAME: {final_answer}")


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: polish-raymond-magda-m
❓ Question: Who did the actor who played Ray in the Polish-language version of Everybody Loves Raymond play in Magda M.? Give only the first name.

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
🌐 Active Search References Extracted:
 📄 Source Verified: Polish Version Everybody Loves Raymond Actor Played Ray (https://www.instagram.com/popular/polish-version-everybody-loves-raymond-actor-played-ray)
 📄 Source Verified: Ray Polish Language Version Everybody Loves Raymond Actor (https://www.instagram.com/popular/ray-polish-language-version-everybody-loves-raymond-actor)
 📄 Source Verified: Ray Romano - Wikipedia (https://en.wikipedia.org/wiki/Ray_Romano)
------------------------------------------------------------

⏳ [Rate Li

In [40]:
if __name__ == "__main__":
    task_17 = {
        'task_id': 'olympics-1928-least-athletes',
        'question': "What country had the least number of athletes at the 1928 Summer Olympics? If there's a tie for a number of athletes, return the first in alphabetical order. Give the IOC country code as your answer."
    }

    # สั่งรันผ่านกระบวนการ ReAct Engine ตัวเดิม
    final_answer = process_task(task_17)
    print(f"\n🚀 FINAL IOC COUNTRY CODE: {final_answer}")


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: olympics-1928-least-athletes
❓ Question: What country had the least number of athletes at the 1928 Summer Olympics? If there's a tie for a number of athletes, return the first in alphabetical order. Give the IOC country code as your answer.

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
🌐 Active Search References Extracted:
 📄 Source Verified: 1928 Summer Olympics - Wikipedia (https://en.wikipedia.org/wiki/1928_Summer_Olympics)
 📄 Source Verified: Athletics at the 1928 Summer Olympics - Wikipedia (https://en.wikipedia.org/wiki/Athletics_at_the_1928_Summer_Olympics)
 📄 Source Verified: Olympic Games of 1928 (Summer) | Women's Studies and Feminism | Research Starters | EBSCO Research (https://www.ebsco.com/research-starters/womens-studies-and-fe

In [58]:
if __name__ == "__main__":
    task_13 = {
        'task_id': 'yankees-1977-stats',
        'question': "How many at bats did the Yankee with the most walks in the 1977 regular season have that same season?"
    }

    # ส่งโจทย์เข้าสู่กระบวนการ ReAct Engine
    final_answer = process_task(task_13)
    print(f"\n🚀 FINAL AT BATS COUNT: {final_answer}")


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: yankees-1977-stats
❓ Question: How many at bats did the Yankee with the most walks in the 1977 regular season have that same season?

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
🌐 Active Search References Extracted:
 📄 Source Verified: 1977 New York Yankees Player Walks Leader (https://www.statmuse.com/mlb/ask/1977-new-york-yankees-player-walks-leader)
 📄 Source Verified: New York Yankees 1977 Team & Player Stats | StatMuse (https://www.statmuse.com/mlb/team/new-york-yankees-76/stats/1977)
 📄 Source Verified: 1977 Yankees Player Hitting Stat Leaders (https://www.mlb.com/yankees/stats/1977)
------------------------------------------------------------

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

🤖 [Node: AGENT] -> Evaluating

In [57]:
if __name__ == "__main__":
    task_15 = {
        'task_id': 'nasa-award-arendt',
        'question': "On June 6, 2023, an article by Carolyn Collins Petersen was published in Universe Today. This article mentions a team that produced a paper... Under what NASA award number was the work performed by R. G. Arendt supported by?"
    }

    # ส่งโจทย์เข้าสู่กระบวนการ ReAct Engine ให้บอทสืบค้นไขว้ระบบ
    final_answer = process_task(task_15)
    print(f"\n🚀 FINAL NASA AWARD NUMBER: {final_answer}")


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: nasa-award-arendt
❓ Question: On June 6, 2023, an article by Carolyn Collins Petersen was published in Universe Today. This article mentions a team that produced a paper... Under what NASA award number was the work performed by R. G. Arendt supported by?

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
🌐 Active Search References Extracted:
 📄 Source Verified: INTERNET OF AGENTS: WEAVING A WEB OF HET (https://proceedings.iclr.cc/paper_files/paper/2025/file/59c27bf8d56d3d50c7aeaf7535dee975-Paper-Conference.pdf)
 📄 Source Verified: Diverse Query Initialization for Agentic Search (https://arxiv.org/pdf/2606.17209)
 📄 Source Verified: Added task files · mark-hug/Final_Assignment_Template at 8eea4d3 (https://huggingface.co/spaces/mark-hug/Final_Assign

In [56]:
if __name__ == "__main__":
    task_6 = {
        'task_id': 'abstract-algebra-commutative',
        'question': """Given this table defining * on the set S = {a, b, c, d, e}

|*|a|b|c|d|e|
|---|---|---|---|---|---|
|a|a|b|c|b|d|
|b|b|c|a|e|c|
|c|c|a|b|b|a|
|d|b|e|b|e|d|
|e|d|b|a|d|c|

provide the subset of S involved in any possible counter-examples that prove * is not commutative. Provide your answer as a comma separated list of the elements in the set in alphabetical order."""
    }

    # ส่งตารางตรรกะคณิตศาสตร์เข้าสู่ ReAct Engine
    final_answer = process_task(task_6)
    print(f"\n🚀 FINAL SUBSET ANSWER: {final_answer}")


[SYSTEM LOG] Model: gemini-3.1-flash-lite
📝 Processing ID: abstract-algebra-commutative
❓ Question: Given this table defining * on the set S = {a, b, c, d, e}

|*|a|b|c|d|e|
|---|---|---|---|---|---|
|a|a|b|c|b|d|
|b|b|c|a|e|c|
|c|c|a|b|b|a|
|d|b|e|b|e|d|
|e|d|b|a|d|c|

provide the subset of S involved in any possible counter-examples that prove * is not commutative. Provide your answer as a comma separated list of the elements in the set in alphabetical order.

 🧠 STARTING REACT LOOP (SYSTEMATIC THOUGHT PROCESS)

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

⚙️  [Node: TOOLS] -> Executing Information Retrieval...
------------------------------------------------------------
📥 Content Ingested Successfully! [Total length: 82 characters]
📄 Preview text:
Analysis: Calculated via programmatic matrix scan.
FINAL_COUNT: 0
Result Set: b, e...
------------------------------------------------------------

⏳ [Rate Limiter] Pacing request to stay under 15 RPM limit...

🤖 [Node: 